# InvariantRRF — Information Fusion strengthening audit

This notebook is a **new scientific strengthening run**, not a rewrite of the rejected manuscript. It is designed to answer the exact objections raised by the *Information Sciences* senior editor before deciding whether a substantially stronger version is suitable for *Information Fusion* or another high-novelty venue.

It tests four questions:

1. **Is the representation-count problem specific to RRF?**  We repeat exact-copy and redundant-family experiments for a family of monotone additive rank-fusion kernels, not only reciprocal rank.
2. **Does the family-budget principle generalize?**  We test the same fixed-provenance-budget construction across those kernels and against hierarchical/nested fusion and score-sum fusion on real families.
3. **Can useful within-family diversity be separated from mere multiplicity?**  We audit a qrels-free, bounded diversity-mass extension over real BM25 and SPLADE families. This is exploratory: the notebook does not force it to succeed.
4. **Are the StableRRF-style prefix bounds merely sufficient, or exact under the stated open-tail uncertainty model?**  We compare the generic bound certificate against exhaustive small-universe completions for several monotone rank kernels and report any false positives or false negatives.

## Required Kaggle inputs

Attach the frozen outputs from the existing study:

- `InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip`
- `InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure.zip` (strongly recommended; SPLADE analyses are skipped if absent)
- the same benchmark bundle containing `mpdr/data/*_mpdr/dev/{queries.jsonl,docs.jsonl,qrels.tsv}`

The notebook **does not retrain the canonical E5 model** and does not modify any frozen source runs.

### Optional real-architecture extension

By default the notebook also *tries* to generate two additional public dense retrievers on SciFact and ArguAna only:

- `sentence-transformers/all-MiniLM-L6-v2`
- `BAAI/bge-small-en-v1.5`

This part needs a Kaggle GPU and model download access. If model download is unavailable, the notebook records the failure and continues with the frozen-run audits.

## Output

The final cell writes:

`InvariantRRF_InformationFusion_Strengthening_Results.zip`

Upload that ZIP back to ChatGPT. Do not manually edit any CSV before returning it.


In [ ]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations, permutations, product
import hashlib, json, math, shutil, zipfile, warnings, time

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy.stats import wilcoxon, spearmanr
except Exception as e:
    raise RuntimeError("scipy is required for this audit") from e

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
OUT = ROOT / 'InvariantRRF_InformationFusion_Strengthening'
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'tables').mkdir(exist_ok=True)
(OUT / 'statistics').mkdir(exist_ok=True)
(OUT / 'optional_runs').mkdir(exist_ok=True)

# Core settings
DEPTH = 100
TOP_K = 10
RBO_P = 0.90
FAMILY_SIZE = 8
PERTURB_RATE = 0.05
BOOTSTRAP_N = 10000
SEED = 20260909

# The optional architecture extension is useful for a stronger Information Fusion submission.
RUN_EXTERNAL_RETRIEVERS = True
EXTERNAL_DATASETS = ['SciFact', 'ArguAna']
EXTERNAL_MODELS = {
    'minilm': 'sentence-transformers/all-MiniLM-L6-v2',
    'bge_small': 'BAAI/bge-small-en-v1.5',
}

# Manual overrides. Leave as None for recursive Kaggle discovery.
CANONICAL_ZIP = None
CANONICAL_ROOT = None
SPLADE_CLOSURE_ZIP = None
SPLADE_CLOSURE_ROOT = None

print('INPUT_ROOT:', INPUT_ROOT)
print('OUT:', OUT)
print('Optional external retrievers:', RUN_EXTERNAL_RETRIEVERS)


## 1. Import and verify the frozen outputs

The integrity check uses the canonical `RUN_MANIFEST.json` when available. No paper-facing ranking is regenerated in this section.


In [ ]:
def sha256_file(path, chunk=1024*1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def discover_zip(keywords):
    zips = list(INPUT_ROOT.rglob('*.zip')) if INPUT_ROOT.exists() else []
    scored = []
    for p in zips:
        n = p.name.lower()
        score = sum(1 for k in keywords if k.lower() in n)
        if score:
            scored.append((score, len(n), str(p), p))
    if not scored:
        return None
    scored.sort(reverse=True)
    return scored[0][-1]

def find_canonical_root(root):
    root = Path(root)
    if (root/'runs').is_dir():
        return root
    for m in sorted(root.rglob('RUN_MANIFEST.json')):
        if (m.parent/'runs').is_dir():
            return m.parent
    for d in sorted(p for p in root.rglob('runs') if p.is_dir()):
        if any(d.glob('*_dense_STRICT_top1000.json')):
            return d.parent
    raise FileNotFoundError(f'No canonical root with runs/ found under {root}')

if CANONICAL_ROOT:
    CANON = find_canonical_root(CANONICAL_ROOT)
else:
    z = Path(CANONICAL_ZIP) if CANONICAL_ZIP else discover_zip(['invariantrrf','canonical','deepdense','taskadaptivek'])
    if z and z.exists():
        ex = ROOT/'_ifusion_canonical_import'
        if ex.exists(): shutil.rmtree(ex)
        ex.mkdir(parents=True)
        with zipfile.ZipFile(z,'r') as zf: zf.extractall(ex)
        CANON = find_canonical_root(ex)
        print('Canonical archive:', z)
    else:
        CANON = find_canonical_root(INPUT_ROOT)

RUNS = CANON/'runs'
manifest_path = CANON/'RUN_MANIFEST.json'
manifest_status = {'present': manifest_path.exists(), 'checked': 0, 'failures': []}
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text('utf-8'))
    for rel, expected in manifest.get('files',{}).items():
        p = CANON/rel
        if not p.exists():
            manifest_status['failures'].append((rel,'MISSING'))
        elif sha256_file(p) != expected:
            manifest_status['failures'].append((rel,'HASH_MISMATCH'))
        manifest_status['checked'] += 1
    if manifest_status['failures']:
        raise AssertionError(f"Canonical manifest failures: {manifest_status['failures'][:10]}")
    print('Canonical manifest: PASS', manifest_status['checked'], 'files')
else:
    print('WARNING: canonical RUN_MANIFEST.json not found')

# SPLADE closure is optional for the notebook to remain runnable without it.
def find_splade_root(root):
    root=Path(root)
    candidates=[]
    if (root/'generated_runs').is_dir(): candidates.append(root)
    candidates += [p.parent for p in root.rglob('SPLADE_V43_CLOSURE_PROTOCOL.json')]
    candidates += [p.parent for p in root.rglob('CLOSURE_MANIFEST.json') if (p.parent/'generated_runs').is_dir()]
    for c in candidates:
        if (c/'generated_runs'/'SciFact_splade_selfdistil_top500.json').exists():
            return c
    return None

SPLADE_CLOSURE = None
if SPLADE_CLOSURE_ROOT:
    SPLADE_CLOSURE = find_splade_root(SPLADE_CLOSURE_ROOT)
else:
    z2 = Path(SPLADE_CLOSURE_ZIP) if SPLADE_CLOSURE_ZIP else discover_zip(['splade','twocheckpoint','closure'])
    if z2 and z2.exists():
        ex2=ROOT/'_ifusion_splade_import'
        if ex2.exists(): shutil.rmtree(ex2)
        ex2.mkdir(parents=True)
        with zipfile.ZipFile(z2,'r') as zf: zf.extractall(ex2)
        SPLADE_CLOSURE=find_splade_root(ex2)
        print('SPLADE closure archive:', z2)
    else:
        SPLADE_CLOSURE=find_splade_root(INPUT_ROOT)

SELF_RUNS = SPLADE_CLOSURE/'generated_runs' if SPLADE_CLOSURE else None
print('CANON:', CANON)
print('SPLADE closure:', SPLADE_CLOSURE if SPLADE_CLOSURE else 'NOT FOUND - SPLADE two-checkpoint analyses will be skipped')
assert RUNS.is_dir()


## 2. Load runs, benchmark qrels, and helper functions

Qrels are used only for post-hoc effectiveness measurements. All replication, family-budget, overlap, and certificate-validity calculations are structural.


In [ ]:
def load_run(path):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    raw=json.loads(path.read_text('utf-8'))
    out={}
    for qid, seq in raw.items():
        norm=[]
        for x in seq:
            if isinstance(x,(list,tuple)):
                norm.append((str(x[0]), float(x[1]) if len(x)>1 else 0.0))
            else:
                norm.append((str(x),0.0))
        ids=[d for d,_ in norm]
        if len(ids)!=len(set(ids)):
            raise AssertionError(f'Duplicate document IDs in {path.name}/{qid}')
        out[str(qid)]=norm
    return out

def run_path(ds, source):
    mapping={
        'bm25': RUNS/f'{ds}_bm25_top1000.json',
        'bm25_lowb': RUNS/f'{ds}_bm25_lowb_top1000.json',
        'bm25_highb': RUNS/f'{ds}_bm25_highb_top1000.json',
        'dense': RUNS/f'{ds}_dense_STRICT_top1000.json',
        'splade_ensemble': RUNS/f'{ds}_splade_ensemble_top500.json',
    }
    if source=='splade_self':
        if SELF_RUNS is None:
            raise FileNotFoundError('SPLADE closure not available')
        return SELF_RUNS/f'{ds}_splade_selfdistil_top500.json'
    return mapping[source]

RUN_CACHE={}
def get_run(ds, source):
    key=(ds,source)
    if key not in RUN_CACHE:
        RUN_CACHE[key]=load_run(run_path(ds,source))
    return RUN_CACHE[key]

DATASET_DIRNAMES={
    'SciFact':'scifact_mpdr',
    'TREC-COVID':'trec-covid_mpdr',
    'FiQA':'fiqa_mpdr',
    'ArguAna':'arguana_mpdr',
}

def find_dataset_root(dirname):
    for p in INPUT_ROOT.rglob(dirname):
        if p.is_dir() and (p/'dev'/'qrels.tsv').exists():
            return p
    return None

def load_qrels(root):
    if root is None: return {}
    out=defaultdict(dict)
    for line in (Path(root)/'dev'/'qrels.tsv').read_text('utf-8').splitlines():
        sp=line.split('\t')
        if len(sp)<2: continue
        qid,did=str(sp[0]),str(sp[1])
        try: rel=float(sp[2]) if len(sp)>=3 and sp[2] else 1.0
        except Exception: rel=1.0
        if rel>0: out[qid][did]=rel
    return dict(out)

DATASET_ROOTS={ds:find_dataset_root(dirname) for ds,dirname in DATASET_DIRNAMES.items()}
QRELS={ds:load_qrels(root) for ds,root in DATASET_ROOTS.items()}
for ds in DATASET_DIRNAMES:
    print(ds, 'qrels=',len(QRELS[ds]), 'root=',DATASET_ROOTS[ds])

DATASETS=['SciFact','TREC-COVID','FiQA','ArguAna']
BASE_RUNS={}
for ds in DATASETS:
    BASE_RUNS[ds]={
        'bm25':get_run(ds,'bm25'),
        'dense':get_run(ds,'dense'),
    }
    assert set(BASE_RUNS[ds]['bm25'])==set(BASE_RUNS[ds]['dense'])
print('Frozen base runs loaded for',DATASETS)


In [ ]:
def prefix_docs(seq, depth):
    return [str(d) for d,_ in seq[:min(int(depth),len(seq))]]

def top_docs(res,k=TOP_K):
    return [d for d,_ in res[:k]]

def ndcg_at_k(docids, rels, k=10):
    if not rels: return np.nan
    dcg=0.0
    for i,d in enumerate(docids[:k],start=1):
        rel=float(rels.get(str(d),0.0))
        dcg += (2.0**rel-1.0)/math.log2(i+1.0)
    ideal=sorted((float(v) for v in rels.values()),reverse=True)[:k]
    if not ideal: return 0.0
    idcg=sum((2.0**rel-1.0)/math.log2(i+2.0) for i,rel in enumerate(ideal))
    return dcg/idcg if idcg>0 else 0.0

def rbo_finite(a,b,p=RBO_P,depth=None):
    a=list(a);b=list(b)
    depth=min(depth or max(len(a),len(b)),max(len(a),len(b)))
    if depth<=0:return 1.0
    A=set();B=set();num=den=0.0
    for d in range(1,depth+1):
        if d<=len(a):A.add(a[d-1])
        if d<=len(b):B.add(b[d-1])
        w=p**(d-1);num+=(len(A&B)/d)*w;den+=w
    return num/den if den else 1.0

def holm_adjust(pvals):
    p=np.asarray(pvals,dtype=float);m=len(p)
    if m==0:return p
    order=np.argsort(p);out=np.empty(m,float);running=0.0
    for j,idx in enumerate(order):
        val=(m-j)*p[idx];running=max(running,val);out[idx]=min(1.0,running)
    return out

def wilcoxon_safe(x):
    x=np.asarray(x,float);x=x[np.isfinite(x)]
    if len(x)==0:return np.nan
    if np.allclose(x,0):return 1.0
    try:return float(wilcoxon(x,zero_method='wilcox',alternative='two-sided').pvalue)
    except Exception:return np.nan

def bootstrap_mean_ci(x,n_boot=BOOTSTRAP_N,seed=SEED):
    x=np.asarray(x,float);x=x[np.isfinite(x)]
    if len(x)==0:return (np.nan,np.nan,np.nan)
    rng=np.random.default_rng(seed)
    vals=np.empty(n_boot,float)
    for i in range(n_boot):
        vals[i]=rng.choice(x,size=len(x),replace=True).mean()
    return float(x.mean()),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

def signature(seq,depth=None):
    ids=prefix_docs(seq,depth or len(seq))
    return hashlib.sha256('\x1f'.join(ids).encode()).hexdigest()

def deterministic_seed(*parts):
    s='||'.join(map(str,parts)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(s).digest()[:8],'big')%(2**32-1)


## 3. General monotone additive rank fusion

Instead of treating RRF as the only fusion rule, define a broad additive rank-scoring class

\[
S(d)=\sum_i w_i\,\phi(r_i(d)),
\]

where \(\phi(r)\) is non-negative and non-increasing with rank. RRF is one member of this class. The empirical audit below uses six qualitatively different rank kernels.

The family-budget construction is applied unchanged: each declared family has one total budget, exact duplicates are canonicalized within the family, and the budget is split across the remaining distinct members.


In [ ]:
def kernel_value(name,r,depth=DEPTH):
    r=float(r);depth=max(1,int(depth))
    if name=='rrf60': return 61.0/(60.0+r)                 # normalized phi(1)=1
    if name=='inverse_rank': return 1.0/r
    if name=='inverse_sqrt': return 1.0/math.sqrt(r)
    if name=='exp20': return math.exp(-(r-1.0)/20.0)
    if name=='log_discount': return 1.0/math.log2(r+1.0)
    if name=='borda': return max(0.0,(depth-r+1.0)/depth)
    raise KeyError(name)

KERNELS=['rrf60','inverse_rank','inverse_sqrt','exp20','log_discount','borda']

def additive_fuse(rankings,depth=DEPTH,kernel='rrf60',weights=None):
    weights=weights or {n:1.0 for n in rankings}
    scores=defaultdict(float)
    for name,seq in rankings.items():
        w=float(weights.get(name,1.0))
        for r,did in enumerate(prefix_docs(seq,depth),start=1):
            scores[did]+=w*kernel_value(kernel,r,depth)
    return sorted(scores.items(),key=lambda x:(-x[1],x[0]))

def unique_family_representatives(rankings,family_map,depth=DEPTH):
    fams=defaultdict(list)
    for name in rankings:fams[str(family_map[name])].append(name)
    reps={};rep_family={}
    for fam,members in fams.items():
        bysig=defaultdict(list)
        for m in members:bysig[signature(rankings[m],depth)].append(m)
        for same in bysig.values():
            rep=sorted(same)[0]
            reps[rep]=rankings[rep];rep_family[rep]=fam
    return reps,rep_family

def family_budget_fuse(rankings,family_map,depth=DEPTH,kernel='rrf60',family_budgets=None):
    reps,rep_family=unique_family_representatives(rankings,family_map,depth)
    fam_members=defaultdict(list)
    for rep,fam in rep_family.items():fam_members[fam].append(rep)
    family_budgets=family_budgets or {fam:1.0 for fam in fam_members}
    weights={}
    for fam,members in fam_members.items():
        b=float(family_budgets.get(fam,1.0))
        for m in members:weights[m]=b/len(members)
    return additive_fuse(reps,depth,kernel,weights)

def nested_family_fuse(family_rankings,outside_rankings,depth=DEPTH,kernel='rrf60'):
    inner=additive_fuse(family_rankings,depth,kernel)
    inner_seq=[(d,s) for d,s in inner[:depth]]
    outer={'FAMILY_CONSENSUS':inner_seq,**outside_rankings}
    return additive_fuse(outer,depth,kernel)

# Basic algebraic smoke test on synthetic lists.
_syn={'a':[('x',1),('y',.5),('z',.1)],'b':[('z',1),('x',.5),('y',.1)]}
for ker in KERNELS:
    base=family_budget_fuse(_syn,{'a':'A','b':'B'},3,ker)
    dup=dict(_syn);dup['a_copy']=_syn['a']
    got=family_budget_fuse(dup,{'a':'A','a_copy':'A','b':'B'},3,ker)
    assert base==got,(ker,base,got)
print('Generic family-budget exact-copy smoke test: PASS for',len(KERNELS),'kernels')


## 4. Cross-kernel exact-copy audit

This directly tests whether source-count sensitivity is an RRF-only artifact. For every dataset, both BM25 and dense evidence are copied once. Each method is compared with its own pre-copy output.


In [ ]:
exact_rows=[]
for ds in DATASETS:
    qids=sorted(set(BASE_RUNS[ds]['bm25'])&set(BASE_RUNS[ds]['dense']))
    for ker in KERNELS:
        for attacked in ['bm25','dense']:
            other='dense' if attacked=='bm25' else 'bm25'
            ord_order=[];ord_set=[];fam_order=[];fam_set=[]
            for q in qids:
                base={attacked:BASE_RUNS[ds][attacked][q],other:BASE_RUNS[ds][other][q]}
                fm={attacked:attacked,other:other}
                ref_o=top_docs(additive_fuse(base,DEPTH,ker))
                ref_f=top_docs(family_budget_fuse(base,fm,DEPTH,ker))
                dup=dict(base);dup[f'{attacked}__copy']=base[attacked]
                fmd=dict(fm);fmd[f'{attacked}__copy']=attacked
                got_o=top_docs(additive_fuse(dup,DEPTH,ker))
                got_f=top_docs(family_budget_fuse(dup,fmd,DEPTH,ker))
                ord_order.append(int(got_o==ref_o));ord_set.append(int(set(got_o)==set(ref_o)))
                fam_order.append(int(got_f==ref_f));fam_set.append(int(set(got_f)==set(ref_f)))
            exact_rows.append({
                'dataset':ds,'kernel':ker,'attacked_source':attacked,'n_queries':len(qids),
                'ordinary_order_preservation':float(np.mean(ord_order)),
                'ordinary_set_preservation':float(np.mean(ord_set)),
                'family_budget_order_preservation':float(np.mean(fam_order)),
                'family_budget_set_preservation':float(np.mean(fam_set)),
            })
exact_df=pd.DataFrame(exact_rows)
exact_df.to_csv(OUT/'tables'/'generalized_exact_copy_audit.csv',index=False)
display(exact_df.groupby('kernel',as_index=False).agg(
    ordinary_order=('ordinary_order_preservation','mean'),
    ordinary_set=('ordinary_set_preservation','mean'),
    family_order=('family_budget_order_preservation','mean'),
    family_set=('family_budget_set_preservation','mean'),
))
assert (exact_df.family_budget_order_preservation==1.0).all()
assert (exact_df.family_budget_set_preservation==1.0).all()
print('Family-budget exact-copy invariance: PASS in',len(exact_df),'dataset/kernel/source conditions')


## 5. Cross-kernel redundant-family scaling

The existing eight-member qrels-free stress test is regenerated deterministically from the frozen parent rankings. This section asks whether the same representation-count effect persists for other additive rank kernels, and whether fixed family mass still removes most of the induced drift.


In [ ]:
def perturb_adjacent(seq,rate,seed,depth=DEPTH):
    docs=prefix_docs(seq,depth);n=len(docs)
    if n<2 or rate<=0:return [(d,0.0) for d in docs]
    rng=np.random.default_rng(seed)
    out=list(docs)
    swaps=max(1,int(round(rate*n)))
    candidates=list(range(n-1));rng.shuffle(candidates)
    used=set();done=0
    for i in candidates:
        if i in used or i+1 in used:continue
        out[i],out[i+1]=out[i+1],out[i]
        used.update([i,i+1]);done+=1
        if done>=swaps:break
    assert set(out)==set(docs)
    return [(d,0.0) for d in out]

def make_family(base_seq,size=FAMILY_SIZE,rate=PERTURB_RATE,key=()):
    fam={'original':base_seq}
    for j in range(1,int(size)):
        fam[f'variant{j}']=perturb_adjacent(base_seq,rate,deterministic_seed(*key,j),DEPTH)
    return fam

scaling_rows=[];scaling_perq=[]
for ds in DATASETS:
    qids=sorted(set(BASE_RUNS[ds]['bm25'])&set(BASE_RUNS[ds]['dense']))
    for ker in KERNELS:
        for attacked in ['bm25','dense']:
            other='dense' if attacked=='bm25' else 'bm25'
            for q in qids:
                ba=BASE_RUNS[ds][attacked][q];bo=BASE_RUNS[ds][other][q]
                clean={attacked:ba,other:bo}
                clean_top=top_docs(additive_fuse(clean,DEPTH,ker),TOP_K)
                clean_n=ndcg_at_k(clean_top,QRELS[ds].get(q,{}),TOP_K)
                fam=make_family(ba,key=(ds,q,attacked,ker,'generalized'))
                expanded={f'{attacked}__{n}':seq for n,seq in fam.items()};expanded[other]=bo
                fam_names=[n for n in expanded if n.startswith(attacked+'__')]
                fmap={n:'ATTACKED' for n in fam_names};fmap[other]='OTHER'
                ordinary=top_docs(additive_fuse(expanded,DEPTH,ker),TOP_K)
                fb=top_docs(family_budget_fuse(expanded,fmap,DEPTH,ker),TOP_K)
                nested=top_docs(nested_family_fuse({n:expanded[n] for n in fam_names},{other:bo},DEPTH,ker),TOP_K)
                on=ndcg_at_k(ordinary,QRELS[ds].get(q,{}),TOP_K)
                fn=ndcg_at_k(fb,QRELS[ds].get(q,{}),TOP_K)
                nn=ndcg_at_k(nested,QRELS[ds].get(q,{}),TOP_K)
                parent=prefix_docs(ba,DEPTH)
                rbos=[rbo_finite(parent,prefix_docs(seq,DEPTH),RBO_P,DEPTH) for n,seq in fam.items() if n!='original']
                scaling_perq.append({
                    'dataset':ds,'kernel':ker,'attacked_source':attacked,'qid':q,
                    'mean_variant_rbo':float(np.mean(rbos)),
                    'ordinary_set_preserved':int(set(ordinary)==set(clean_top)),
                    'family_budget_set_preserved':int(set(fb)==set(clean_top)),
                    'nested_set_preserved':int(set(nested)==set(clean_top)),
                    'ordinary_abs_ndcg_drift':abs(on-clean_n) if np.isfinite(clean_n) else np.nan,
                    'family_budget_abs_ndcg_drift':abs(fn-clean_n) if np.isfinite(clean_n) else np.nan,
                    'nested_abs_ndcg_drift':abs(nn-clean_n) if np.isfinite(clean_n) else np.nan,
                })

spq=pd.DataFrame(scaling_perq)
for keys,g in spq.groupby(['dataset','kernel','attacked_source'],sort=True):
    ds,ker,src=keys
    diff=(g.ordinary_abs_ndcg_drift-g.family_budget_abs_ndcg_drift).to_numpy(float)
    mean,lo,hi=bootstrap_mean_ci(diff,seed=deterministic_seed(ds,ker,src,'boot'))
    p=wilcoxon_safe(diff)
    scaling_rows.append({
        'dataset':ds,'kernel':ker,'attacked_source':src,'n_queries':len(g),
        'mean_family_rbo':g.mean_variant_rbo.mean(),
        'ordinary_set':g.ordinary_set_preserved.mean(),
        'family_budget_set':g.family_budget_set_preserved.mean(),
        'nested_set':g.nested_set_preserved.mean(),
        'ordinary_abs_drift':g.ordinary_abs_ndcg_drift.mean(),
        'family_budget_abs_drift':g.family_budget_abs_ndcg_drift.mean(),
        'nested_abs_drift':g.nested_abs_ndcg_drift.mean(),
        'ordinary_minus_family_drift':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,
    })
scaling_df=pd.DataFrame(scaling_rows)
scaling_df['p_holm_all']=holm_adjust(scaling_df.p_raw.fillna(1).to_numpy(float))
spq.to_csv(OUT/'tables'/'generalized_scaling_per_query.csv',index=False)
scaling_df.to_csv(OUT/'tables'/'generalized_scaling_summary.csv',index=False)
display(scaling_df.groupby('kernel',as_index=False).agg(
    ordinary_set=('ordinary_set','mean'),family_budget_set=('family_budget_set','mean'),nested_set=('nested_set','mean'),
    ordinary_drift=('ordinary_abs_drift','mean'),family_budget_drift=('family_budget_abs_drift','mean'),nested_drift=('nested_abs_drift','mean'),
    drift_reduction=('ordinary_minus_family_drift','mean')
))


## 6. Real-family fusion alternatives

Synthetic families isolate the mechanism. This section uses only real families already present in the frozen outputs:

- three BM25 parameterizations on SciFact, FiQA, and ArguAna;
- EnsembleDistil + SelfDistil SPLADE checkpoints on SciFact and ArguAna when the closure archive is attached.

For each additive rank kernel we compare ordinary flat fusion, fixed family-budget fusion, plain nested fusion, and a representative-only family control. We also include min-max score-sum fusion on these real scored runs to test whether multiplicity sensitivity is limited to rank-only fusion.


In [ ]:
REAL_SCENARIOS=[]
for ds in ['SciFact','FiQA','ArguAna']:
    REAL_SCENARIOS.append({
        'scenario':'BM25-family','dataset':ds,
        'family_sources':['bm25','bm25_lowb','bm25_highb'],
        'representative':'bm25',
        'outside_sources':(['splade_ensemble'] if (RUNS/f'{ds}_splade_ensemble_top500.json').exists() else [])+['dense']
    })
if SELF_RUNS is not None:
    for ds in ['SciFact','ArguAna']:
        if (SELF_RUNS/f'{ds}_splade_selfdistil_top500.json').exists() and (RUNS/f'{ds}_splade_ensemble_top500.json').exists():
            REAL_SCENARIOS.append({
                'scenario':'SPLADE-family','dataset':ds,
                'family_sources':['splade_ensemble','splade_self'],
                'representative':'splade_ensemble',
                'outside_sources':['bm25']
            })
print('Real scenarios:',[(x['scenario'],x['dataset']) for x in REAL_SCENARIOS])

def minmax_scores(seq,depth=DEPTH):
    vals=[float(s) for _,s in seq[:depth]]
    if not vals:return {}
    lo,hi=min(vals),max(vals)
    if not np.isfinite(lo) or not np.isfinite(hi):return {}
    if abs(hi-lo)<1e-15:
        # deterministic rank fallback when a source has constant scores
        return {str(d):(depth-r+1)/depth for r,(d,_) in enumerate(seq[:depth],start=1)}
    return {str(d):(float(s)-lo)/(hi-lo) for d,s in seq[:depth]}

def combsum(rankings,depth=DEPTH,weights=None):
    weights=weights or {n:1.0 for n in rankings}
    scores=defaultdict(float)
    for n,seq in rankings.items():
        w=float(weights.get(n,1.0))
        for d,s in minmax_scores(seq,depth).items():scores[d]+=w*s
    return sorted(scores.items(),key=lambda x:(-x[1],x[0]))

def family_weights_from_map(rankings,fmap,depth=DEPTH):
    reps,rep_family=unique_family_representatives(rankings,fmap,depth)
    members=defaultdict(list)
    for r,f in rep_family.items():members[f].append(r)
    w={}
    for f,ms in members.items():
        for m in ms:w[m]=1.0/len(ms)
    return reps,w

real_rows=[];score_rows=[]
for sc in REAL_SCENARIOS:
    ds=sc['dataset'];fam=sc['family_sources'];rep=sc['representative'];outside=sc['outside_sources']
    runs={s:get_run(ds,s) for s in set(fam+outside)}
    qids=sorted(set.intersection(*(set(runs[s]) for s in runs)))
    for q in qids:
        family={s:runs[s][q] for s in fam};outs={s:runs[s][q] for s in outside}
        expanded={**family,**outs};clean={rep:runs[rep][q],**outs}
        fmap={s:'FAMILY' for s in fam};fmap.update({s:f'OUT::{s}' for s in outside})
        rels=QRELS[ds].get(q,{})
        for ker in KERNELS:
            clean_top=top_docs(additive_fuse(clean,DEPTH,ker))
            ordinary=top_docs(additive_fuse(expanded,DEPTH,ker))
            fb=top_docs(family_budget_fuse(expanded,fmap,DEPTH,ker))
            nested=top_docs(nested_family_fuse(family,outs,DEPTH,ker))
            cn=ndcg_at_k(clean_top,rels,TOP_K);on=ndcg_at_k(ordinary,rels,TOP_K);fn=ndcg_at_k(fb,rels,TOP_K);nn=ndcg_at_k(nested,rels,TOP_K)
            real_rows.append({
                'scenario':sc['scenario'],'dataset':ds,'qid':q,'kernel':ker,
                'ordinary_set_vs_clean':int(set(ordinary)==set(clean_top)),
                'family_budget_set_vs_clean':int(set(fb)==set(clean_top)),
                'nested_set_vs_clean':int(set(nested)==set(clean_top)),
                'ordinary_delta_ndcg':on-cn if np.isfinite(cn) else np.nan,
                'family_budget_delta_ndcg':fn-cn if np.isfinite(cn) else np.nan,
                'nested_delta_ndcg':nn-cn if np.isfinite(cn) else np.nan,
                'family_budget_minus_ordinary_ndcg':fn-on if np.isfinite(on) and np.isfinite(fn) else np.nan,
            })
        # Score-sum audit once per query.
        clean_score=top_docs(combsum(clean,DEPTH))
        expanded_score=top_docs(combsum(expanded,DEPTH))
        reps,w=family_weights_from_map(expanded,fmap,DEPTH)
        fb_score=top_docs(combsum(reps,DEPTH,w))
        cn=ndcg_at_k(clean_score,rels,TOP_K);on=ndcg_at_k(expanded_score,rels,TOP_K);fn=ndcg_at_k(fb_score,rels,TOP_K)
        score_rows.append({
            'scenario':sc['scenario'],'dataset':ds,'qid':q,
            'ordinary_set_vs_clean':int(set(expanded_score)==set(clean_score)),
            'family_budget_set_vs_clean':int(set(fb_score)==set(clean_score)),
            'ordinary_delta_ndcg':on-cn if np.isfinite(cn) else np.nan,
            'family_budget_delta_ndcg':fn-cn if np.isfinite(cn) else np.nan,
        })

real_perq=pd.DataFrame(real_rows);score_perq=pd.DataFrame(score_rows)
real_summary=real_perq.groupby(['scenario','dataset','kernel'],as_index=False).agg(
    ordinary_set=('ordinary_set_vs_clean','mean'),family_budget_set=('family_budget_set_vs_clean','mean'),nested_set=('nested_set_vs_clean','mean'),
    ordinary_delta=('ordinary_delta_ndcg','mean'),family_budget_delta=('family_budget_delta_ndcg','mean'),nested_delta=('nested_delta_ndcg','mean'),
    fb_minus_ordinary=('family_budget_minus_ordinary_ndcg','mean'))
score_summary=score_perq.groupby(['scenario','dataset'],as_index=False).agg(
    ordinary_set=('ordinary_set_vs_clean','mean'),family_budget_set=('family_budget_set_vs_clean','mean'),
    ordinary_delta=('ordinary_delta_ndcg','mean'),family_budget_delta=('family_budget_delta_ndcg','mean'))
real_perq.to_csv(OUT/'tables'/'real_family_additive_per_query.csv',index=False)
real_summary.to_csv(OUT/'tables'/'real_family_additive_summary.csv',index=False)
score_perq.to_csv(OUT/'tables'/'real_family_combsum_per_query.csv',index=False)
score_summary.to_csv(OUT/'tables'/'real_family_combsum_summary.csv',index=False)
display(real_summary)
display(score_summary)


## 7. Redundancy-to-complementarity audit and bounded diversity-mass candidate

The SPLADE boundary suggests that **shared provenance is not the same as useless duplication**. To investigate that directly, this section computes a qrels-free family diversity diameter

\[
D_g = \max_{i,j\in G_g} (1-\mathrm{RBO}(R_i,R_j)),
\]

on the observed prefix. Exact copies do not change this diameter. We then explore, without selecting a winner in advance, a bounded family budget

\[
W_g^{\mathrm{eff}} = W_g(1+\lambda D_g), \qquad \lambda\in\{0,.25,.5,.75,1\}.
\]

The budget can increase when a family is genuinely diverse, but never merely because more copies are supplied; for \(\lambda\le1\), the uplift is at most 2x. This is an **exploratory candidate**, not a claimed final method. Its purpose is to test whether the ArguAna SPLADE relevance boundary can be addressed without sacrificing exact-copy neutrality.


In [ ]:
LAMBDA_GRID=[0.0,0.25,0.5,0.75,1.0]

def family_diversity_diameter(family_rankings,depth=DEPTH):
    # collapse exact prefix duplicates before measuring diameter
    uniq={}
    for n,seq in family_rankings.items():uniq.setdefault(signature(seq,depth),(n,seq))
    seqs=[prefix_docs(seq,depth) for _,seq in uniq.values()]
    if len(seqs)<=1:return 0.0
    return float(max(1.0-rbo_finite(a,b,RBO_P,depth) for a,b in combinations(seqs,2)))

def diversity_bounded_fuse(rankings,fmap,family_name,lam,depth=DEPTH,kernel='rrf60'):
    fam_rankings={n:rankings[n] for n in rankings if fmap[n]==family_name}
    D=family_diversity_diameter(fam_rankings,depth)
    budgets={f:1.0 for f in set(fmap.values())}
    budgets[family_name]=1.0+float(lam)*D
    return family_budget_fuse(rankings,fmap,depth,kernel,budgets),D,budgets[family_name]

div_rows=[]
for sc in REAL_SCENARIOS:
    ds=sc['dataset'];fam=sc['family_sources'];rep=sc['representative'];outside=sc['outside_sources']
    runs={s:get_run(ds,s) for s in set(fam+outside)}
    qids=sorted(set.intersection(*(set(runs[s]) for s in runs)))
    for q in qids:
        family={s:runs[s][q] for s in fam};outs={s:runs[s][q] for s in outside};expanded={**family,**outs};clean={rep:runs[rep][q],**outs}
        fmap={s:'FAMILY' for s in fam};fmap.update({s:f'OUT::{s}' for s in outside})
        clean_top=top_docs(additive_fuse(clean,DEPTH,'rrf60'));rels=QRELS[ds].get(q,{})
        clean_n=ndcg_at_k(clean_top,rels,TOP_K)
        ordinary=top_docs(additive_fuse(expanded,DEPTH,'rrf60'));ordinary_n=ndcg_at_k(ordinary,rels,TOP_K)
        mc=top_docs(family_budget_fuse(expanded,fmap,DEPTH,'rrf60'));mc_n=ndcg_at_k(mc,rels,TOP_K)
        D=family_diversity_diameter(family,DEPTH)
        for lam in LAMBDA_GRID:
            res,D2,budget=diversity_bounded_fuse(expanded,fmap,'FAMILY',lam,DEPTH,'rrf60')
            top=top_docs(res);n=ndcg_at_k(top,rels,TOP_K)
            div_rows.append({
                'scenario':sc['scenario'],'dataset':ds,'qid':q,'lambda':lam,'diversity_diameter':D2,'family_budget':budget,
                'set_vs_clean':int(set(top)==set(clean_top)),
                'ndcg_delta_vs_clean':n-clean_n if np.isfinite(clean_n) else np.nan,
                'ndcg_vs_expanded_ordinary':n-ordinary_n if np.isfinite(n) and np.isfinite(ordinary_n) else np.nan,
                'ordinary_delta_vs_clean':ordinary_n-clean_n if np.isfinite(clean_n) else np.nan,
                'mc_delta_vs_clean':mc_n-clean_n if np.isfinite(clean_n) else np.nan,
            })

div_perq=pd.DataFrame(div_rows)
div_summary=div_perq.groupby(['scenario','dataset','lambda'],as_index=False).agg(
    mean_diversity=('diversity_diameter','mean'),mean_budget=('family_budget','mean'),set_vs_clean=('set_vs_clean','mean'),
    ndcg_delta_vs_clean=('ndcg_delta_vs_clean','mean'),ndcg_vs_expanded_ordinary=('ndcg_vs_expanded_ordinary','mean'))
# Correlation uses lambda=0 rows only to avoid duplicating each query five times.
base_div=div_perq[div_perq['lambda']==0].copy()
corr_rows=[]
for (sc,ds),g in base_div.groupby(['scenario','dataset']):
    x=g.diversity_diameter.to_numpy(float);y=g.ordinary_delta_vs_clean.to_numpy(float)
    mask=np.isfinite(x)&np.isfinite(y)
    if mask.sum()>=3:
        rho,p=spearmanr(x[mask],y[mask])
    else:rho,p=np.nan,np.nan
    corr_rows.append({'scenario':sc,'dataset':ds,'n':int(mask.sum()),'spearman_diversity_vs_marginal_utility':rho,'p_raw':p})
corr_df=pd.DataFrame(corr_rows)
if len(corr_df):corr_df['p_holm']=holm_adjust(corr_df.p_raw.fillna(1).to_numpy(float))

div_perq.to_csv(OUT/'tables'/'diversity_bounded_mass_per_query.csv',index=False)
div_summary.to_csv(OUT/'tables'/'diversity_bounded_mass_summary.csv',index=False)
corr_df.to_csv(OUT/'statistics'/'diversity_utility_correlation.csv',index=False)
display(div_summary)
display(corr_df)

# Structural guard: exact copies remain neutral under every lambda.
for ker in ['rrf60']:
    base={'a':[('x',1),('y',.5),('z',0)],'b':[('z',1),('x',.5),('y',0)]}
    fmap={'a':'FAMILY','b':'OUT'}
    for lam in LAMBDA_GRID:
        r1=diversity_bounded_fuse(base,fmap,'FAMILY',lam,3,ker)[0]
        dup={**base,'a_copy':base['a']};fm2={**fmap,'a_copy':'FAMILY'}
        r2=diversity_bounded_fuse(dup,fm2,'FAMILY',lam,3,ker)[0]
        assert r1==r2,(lam,r1,r2)
print('Diversity-bounded candidate exact-copy neutrality: PASS for all lambdas')


## 8. Stable prefix certification: exhaustive exactness audit for a general additive kernel

The current paper presents the prefix certificate as sufficient. Under an **open-tail model** in which any unseen document may appear at any later rank or remain absent from a source's returned list, the natural upper bound for a missing contribution is \(w_i\phi(L_i+1)\).

This section exhaustively enumerates all ordered subsets of the unseen documents for small universes. It compares the bound certificate against the exact set/order stability observed over **every admissible completion**. Any false positive or false negative is saved as a counterexample.

This is the most important theoretical diagnostic in the notebook. If the bound certificate exactly matches exhaustive completion stability across the tested kernels, we can attempt a formal necessity-and-sufficiency proof in the next manuscript rather than merely adding more equations.


In [ ]:
def ordered_subsets(items):
    items=tuple(items)
    out=[()]
    for r in range(1,len(items)+1):out.extend(permutations(items,r))
    return out

def generic_bounds(prefixes,weights,kernel,universe):
    observed=set().union(*(set(p) for p in prefixes.values())) if prefixes else set()
    LB={d:0.0 for d in observed}
    UB={d:0.0 for d in observed}
    unseen_bound=0.0
    for s,pref in prefixes.items():
        w=float(weights[s]);L=len(pref)
        pos={d:r for r,d in enumerate(pref,start=1)}
        tail=w*kernel_value(kernel,L+1,max(L+1,len(universe)))
        unseen_bound+=tail
        for d in observed:
            if d in pos:
                v=w*kernel_value(kernel,pos[d],max(L+1,len(universe)))
                LB[d]+=v;UB[d]+=v
            else:
                UB[d]+=tail
    return LB,UB,unseen_bound,observed

def bound_certificate(prefixes,weights,kernel,universe,K):
    LB,UB,U,observed=generic_bounds(prefixes,weights,kernel,universe)
    if len(observed)<K:return False,False,()
    ordered=sorted(observed,key=lambda d:(-LB[d],d))
    top=ordered[:K]
    outsider=[d for d in observed if d not in set(top)]
    max_out=max([UB[d] for d in outsider]+[U])
    set_cert=min(LB[d] for d in top)>max_out
    order_cert=True
    for j,d in enumerate(top):
        later=top[j+1:]+outsider
        mx=max([UB[x] for x in later]+[U])
        if not (LB[d]>mx):
            order_cert=False;break
    return bool(set_cert),bool(order_cert),tuple(top)

def fuse_complete(rankings,weights,kernel,universe):
    scores={d:0.0 for d in universe}
    depth=max(len(universe),1)
    for s,seq in rankings.items():
        w=float(weights[s])
        for r,d in enumerate(seq,start=1):scores[d]+=w*kernel_value(kernel,r,depth)
    ordered=sorted(universe,key=lambda d:(-scores[d],d))
    return ordered,scores

def brute_stability(prefixes,weights,kernel,universe,K):
    per_source=[];sources=list(prefixes)
    for s in sources:
        rem=[d for d in universe if d not in prefixes[s]]
        per_source.append([tuple(prefixes[s])+x for x in ordered_subsets(rem)])
    top_sets=set();top_orders=set();strict_set=True;strict_order=True;ncomp=0
    for combo in product(*per_source):
        rankings={s:list(seq) for s,seq in zip(sources,combo)}
        order,scores=fuse_complete(rankings,weights,kernel,universe)
        top=tuple(order[:K]);top_orders.add(top);top_sets.add(frozenset(top));ncomp+=1
        vals=[scores[d] for d in order]
        if K<len(order) and not (vals[K-1]>vals[K]+1e-14):strict_set=False
        for j in range(min(K-1,len(order)-1)):
            if not (vals[j]>vals[j+1]+1e-14):strict_order=False
    return (len(top_sets)==1 and strict_set),(len(top_orders)==1 and strict_order),ncomp

rng=np.random.default_rng(SEED)
audit=[];counter=[]
N_CASES=240
for case in range(N_CASES):
    n_docs=int(rng.integers(4,6));n_sources=int(rng.integers(2,4));K=int(rng.integers(1,min(3,n_docs)))
    universe=[f'd{i}' for i in range(n_docs)]
    prefixes={};weights={}
    for si in range(n_sources):
        s=f's{si}';perm=list(rng.permutation(universe));L=int(rng.integers(1,min(3,n_docs)))
        prefixes[s]=perm[:L];weights[s]=float(rng.uniform(.3,1.7))
    ker=KERNELS[case%len(KERNELS)]
    cert_set,cert_order,cert_top=bound_certificate(prefixes,weights,ker,universe,K)
    true_set,true_order,ncomp=brute_stability(prefixes,weights,ker,universe,K)
    row={'case':case,'kernel':ker,'n_docs':n_docs,'n_sources':n_sources,'K':K,'completions':ncomp,
         'certificate_set':cert_set,'true_set_stable':true_set,'certificate_order':cert_order,'true_order_stable':true_order}
    audit.append(row)
    if cert_set!=true_set or cert_order!=true_order:
        counter.append({**row,'prefixes':json.dumps(prefixes),'weights':json.dumps(weights),'certificate_top':json.dumps(cert_top)})

audit_df=pd.DataFrame(audit);counter_df=pd.DataFrame(counter)
audit_df.to_csv(OUT/'tables'/'generic_certificate_exhaustive_audit.csv',index=False)
counter_df.to_csv(OUT/'tables'/'generic_certificate_counterexamples.csv',index=False)
summary=audit_df.groupby('kernel',as_index=False).agg(
    cases=('case','count'),
    set_false_positive=('certificate_set',lambda s:0),
)
# explicit confusion counts
rows=[]
for ker,g in audit_df.groupby('kernel'):
    rows.append({
        'kernel':ker,'cases':len(g),
        'set_false_positive':int(((g.certificate_set==1)&(g.true_set_stable==0)).sum()),
        'set_false_negative':int(((g.certificate_set==0)&(g.true_set_stable==1)).sum()),
        'order_false_positive':int(((g.certificate_order==1)&(g.true_order_stable==0)).sum()),
        'order_false_negative':int(((g.certificate_order==0)&(g.true_order_stable==1)).sum()),
    })
exactness_summary=pd.DataFrame(rows)
exactness_summary.to_csv(OUT/'tables'/'generic_certificate_exactness_summary.csv',index=False)
display(exactness_summary)
if len(counter_df):
    print('IMPORTANT: counterexamples found:',len(counter_df))
    display(counter_df.head(10))
else:
    print('No certificate/exhaustive-stability disagreement in',N_CASES,'random exhaustive cases.')


## 9. Optional additional retrieval architectures

This is deliberately separated from the main audit so a model-download failure cannot invalidate the frozen-run experiments. It adds two independent dense retrieval architectures on the two smaller collections, SciFact and ArguAna, and evaluates their retrieval effectiveness plus multi-source RRF combinations.

These runs are **not used to define provenance families**. They serve as genuinely distinct evidence sources and broaden the retrieval-architecture coverage requested by the editor.


In [ ]:
external_status=[];external_summary=[]
if RUN_EXTERNAL_RETRIEVERS:
    try:
        import torch
        from sentence_transformers import SentenceTransformer

        def read_jsonl(path):
            rows=[]
            with open(path,'r',encoding='utf-8') as f:
                for line in f:
                    if line.strip():rows.append(json.loads(line))
            return rows

        def load_text_maps(root):
            qs=read_jsonl(Path(root)/'dev'/'queries.jsonl');docs=read_jsonl(Path(root)/'dev'/'docs.jsonl')
            qmap={str(x['id']):x.get('text','') for x in qs};dmap={str(x['id']):x.get('text','') for x in docs}
            return qmap,dmap

        def dense_rank_st(qmap,dmap,model_name,top_k=1000,batch=64):
            model=SentenceTransformer(model_name)
            qids=sorted(qmap);docids=list(dmap)
            qtexts=[qmap[q] for q in qids];dtexts=[dmap[d] for d in docids]
            if 'bge-' in model_name.lower():
                qtexts=['Represent this sentence for searching relevant passages: '+x for x in qtexts]
            Q=model.encode(qtexts,batch_size=batch,convert_to_tensor=True,normalize_embeddings=True,show_progress_bar=True)
            D=model.encode(dtexts,batch_size=batch,convert_to_tensor=True,normalize_embeddings=True,show_progress_bar=True)
            K=min(top_k,len(docids));run={}
            for start in range(0,len(qids),128):
                scores=Q[start:start+128]@D.T
                vals,idx=torch.topk(scores,k=K,dim=1)
                vals=vals.detach().cpu().numpy();idx=idx.detach().cpu().numpy()
                for ii,q in enumerate(qids[start:start+len(vals)]):
                    run[q]=[(str(docids[j]),float(vals[ii,r])) for r,j in enumerate(idx[ii])]
            del model,Q,D
            if torch.cuda.is_available():torch.cuda.empty_cache()
            return run

        for ds in EXTERNAL_DATASETS:
            root=DATASET_ROOTS.get(ds)
            if root is None:
                external_status.append({'dataset':ds,'model':'ALL','status':'SKIPPED_NO_BENCHMARK_ROOT','message':''})
                continue
            qmap,dmap=load_text_maps(root);qids=sorted(set(QRELS[ds])&set(qmap))
            qmap={q:qmap[q] for q in qids}
            for short,model_name in EXTERNAL_MODELS.items():
                try:
                    t0=time.time();run=dense_rank_st(qmap,dmap,model_name,top_k=1000)
                    path=OUT/'optional_runs'/f'{ds}_{short}_top1000.json';path.write_text(json.dumps(run),encoding='utf-8')
                    nd=np.mean([ndcg_at_k(prefix_docs(run[q],TOP_K),QRELS[ds][q],TOP_K) for q in qids])
                    external_summary.append({'dataset':ds,'source':short,'model':model_name,'n_queries':len(qids),'ndcg10':nd})
                    external_status.append({'dataset':ds,'model':model_name,'status':'PASS','message':f'{time.time()-t0:.1f}s'})
                except Exception as e:
                    external_status.append({'dataset':ds,'model':model_name,'status':'FAILED','message':repr(e)[:500]})

            # Evaluate any successful new sources as genuinely distinct additions to BM25+dense.
            available={}
            for short in EXTERNAL_MODELS:
                p=OUT/'optional_runs'/f'{ds}_{short}_top1000.json'
                if p.exists():available[short]=load_run(p)
            if available:
                bm=BASE_RUNS[ds]['bm25'];de=BASE_RUNS[ds]['dense']
                for short,rr in available.items():
                    common=sorted(set(qids)&set(bm)&set(de)&set(rr));deltas=[]
                    for q in common:
                        base=top_docs(additive_fuse({'bm25':bm[q],'dense':de[q]},DEPTH,'rrf60'))
                        ext=top_docs(additive_fuse({'bm25':bm[q],'dense':de[q],short:rr[q]},DEPTH,'rrf60'))
                        deltas.append(ndcg_at_k(ext,QRELS[ds][q],TOP_K)-ndcg_at_k(base,QRELS[ds][q],TOP_K))
                    external_summary.append({'dataset':ds,'source':f'BM25+dense+{short}','model':'RRF60 distinct-source addition','n_queries':len(common),'ndcg10':np.mean(deltas)})
    except Exception as e:
        external_status.append({'dataset':'ALL','model':'ALL','status':'FAILED_IMPORT_OR_SETUP','message':repr(e)[:500]})
else:
    external_status.append({'dataset':'ALL','model':'ALL','status':'DISABLED','message':''})

external_status_df=pd.DataFrame(external_status);external_summary_df=pd.DataFrame(external_summary)
external_status_df.to_csv(OUT/'tables'/'external_retriever_status.csv',index=False)
external_summary_df.to_csv(OUT/'tables'/'external_retriever_summary.csv',index=False)
display(external_status_df)
if len(external_summary_df):display(external_summary_df)


## 10. Decision report and immutable result package

The notebook does not declare the new paper successful merely because it ran. The report below summarizes the actual outcomes that matter for the next scientific decision.


In [ ]:
# Decision-facing summaries
kernel_macro=scaling_df.groupby('kernel',as_index=False).agg(
    conditions=('dataset','count'),
    ordinary_set=('ordinary_set','mean'),
    family_budget_set=('family_budget_set','mean'),
    nested_set=('nested_set','mean'),
    ordinary_drift=('ordinary_abs_drift','mean'),
    family_budget_drift=('family_budget_abs_drift','mean'),
    nested_drift=('nested_abs_drift','mean'),
    drift_reduction=('ordinary_minus_family_drift','mean'),
    holm_significant=('p_holm_all',lambda x:int((x<.05).sum())),
)
kernel_macro.to_csv(OUT/'tables'/'generalized_kernel_macro.csv',index=False)

# How many kernels show ordinary exact-copy failure somewhere?
fail_by_kernel=exact_df.groupby('kernel').ordinary_order_preservation.min()<1.0
fb_exact=bool((exact_df.family_budget_order_preservation==1.0).all())

# Diversity candidate: for each real scenario, find descriptive best lambda for nDCG vs clean and for set preservation.
div_dec=[]
for (sc,ds),g in div_summary.groupby(['scenario','dataset']):
    best_nd=g.loc[g.ndcg_delta_vs_clean.idxmax()]
    best_set=g.loc[g.set_vs_clean.idxmax()]
    div_dec.append({'scenario':sc,'dataset':ds,'best_lambda_by_mean_ndcg':best_nd['lambda'],'best_mean_ndcg_delta':best_nd.ndcg_delta_vs_clean,
                    'best_lambda_by_set':best_set['lambda'],'best_set_preservation':best_set.set_vs_clean})
div_dec_df=pd.DataFrame(div_dec)
div_dec_df.to_csv(OUT/'tables'/'diversity_candidate_decision_summary.csv',index=False)

fp=int(((audit_df.certificate_set==1)&(audit_df.true_set_stable==0)).sum()+((audit_df.certificate_order==1)&(audit_df.true_order_stable==0)).sum())
fn=int(((audit_df.certificate_set==0)&(audit_df.true_set_stable==1)).sum()+((audit_df.certificate_order==0)&(audit_df.true_order_stable==1)).sum())

lines=[
    '# Information Fusion strengthening audit — decision report','',
    '## Frozen-input integrity',
    f'- Canonical manifest present: {manifest_status["present"]}',
    f'- Canonical manifest files checked: {manifest_status["checked"]}',
    f'- Manifest failures: {len(manifest_status["failures"])}','',
    '## Generality beyond RRF',
    f'- Additive rank kernels tested: {len(KERNELS)} ({", ".join(KERNELS)})',
    f'- Kernels where ordinary fusion fails exact-copy order preservation in at least one dataset/source condition: {int(fail_by_kernel.sum())}/{len(KERNELS)}',
    f'- Family-budget exact-copy invariant in every tested dataset/kernel/source condition: {fb_exact}',
]
for r in kernel_macro.itertuples():
    lines.append(f'- {r.kernel}: ordinary set={r.ordinary_set:.4f}; family-budget set={r.family_budget_set:.4f}; nested set={r.nested_set:.4f}; ordinary drift={r.ordinary_drift:.6f}; family-budget drift={r.family_budget_drift:.6f}; mean reduction={r.drift_reduction:.6f}')
lines += ['', '## Real-family and complementarity boundary']
for r in div_dec_df.itertuples():
    lines.append(f'- {r.scenario}/{r.dataset}: descriptive best lambda by mean nDCG={r.best_lambda_by_mean_ndcg:g} (delta={r.best_mean_ndcg_delta:+.6f}); best lambda by set={r.best_lambda_by_set:g} (set={r.best_set_preservation:.4f})')
lines += ['', '## Stable prefix-certificate exactness audit',
          f'- Exhaustive random small-universe cases: {len(audit_df)}',
          f'- Total false-positive disagreements (set + order): {fp}',
          f'- Total false-negative disagreements (set + order): {fn}',
          f'- Counterexample rows saved: {len(counter_df)}']
lines += ['', '## Optional architecture extension']
for r in external_status_df.itertuples():
    lines.append(f'- {r.dataset}/{r.model}: {r.status} {r.message}')
lines += ['', '## Interpretation rules',
          '- Do not promote the diversity-bounded mass candidate solely because one lambda improves one dataset.',
          '- If Stable certificate false negatives appear, inspect the saved counterexamples before claiming necessity/sufficiency.',
          '- If the generalized family-budget effect holds across kernels, the next paper can be framed as a property of additive rank fusion rather than an RRF-only patch.',
          '- External-retriever failures caused by unavailable model downloads are infrastructure failures, not scientific outcomes.']
report='\n'.join(lines)
(OUT/'DECISION_REPORT.md').write_text(report,encoding='utf-8')
print(report)

protocol={
    'created_for':'post-INS strengthening / Information Fusion decision',
    'depth':DEPTH,'top_k':TOP_K,'rbo_p':RBO_P,'family_size':FAMILY_SIZE,'perturb_rate':PERTURB_RATE,
    'kernels':KERNELS,'lambda_grid':LAMBDA_GRID,'stable_exhaustive_cases':N_CASES,
    'external_retrievers_enabled':RUN_EXTERNAL_RETRIEVERS,'external_models':EXTERNAL_MODELS,
    'canonical_root_name':CANON.name,'splade_closure_available':SPLADE_CLOSURE is not None,
}
(OUT/'PROTOCOL.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')

# Manifest over outputs before adding the manifest itself and ZIP.
manifest_out={}
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='OUTPUT_SHA256.json':manifest_out[str(p.relative_to(OUT))]=sha256_file(p)
(OUT/'OUTPUT_SHA256.json').write_text(json.dumps(manifest_out,indent=2),encoding='utf-8')

zip_path=ROOT/'InvariantRRF_InformationFusion_Strengthening_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():zf.write(p,arcname=str(p.relative_to(OUT)))
print('\nUPLOAD BACK TO CHATGPT:')
print(zip_path)
print('SHA256:',sha256_file(zip_path))
